# DADA3: 2 - The team

Six TODOs, 9 points.

Notebook 01 built a pair that works on one bug. This notebook builds the team around it: triage decides which issues are bugs, one pair per bug runs in parallel, and a report brings their results together for the maintainer.

The top half of the diagram is what you build here.

![architecture](figures/architecture.png)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import importlib, json, os
from bugteam import config, repo, team

issues = repo.load_issues()
print(config.CHAT_BACKEND, config.model_name(), "| sandbox:", config.SANDBOX, "| MAX_PARALLEL:", config.MAX_PARALLEL)

## 1. Triage

Not every issue is a bug. Three of the ten in this repository are not: in two, the code does what its docstring says and the reporter expected something else, and one asks for a new feature. A team that "fixed" those would change documented behavior that other code may depend on. So the first step is triage, and only the issues it labels as bugs get a pair.

Triage makes one structured-output call per issue, the way `check` in W07b made one call per sentence. Each call gets the library's code and one issue, and returns a `Label`. Read `Label` and `TRIAGE_PROMPT` in `team.py`. Two details in them matter.

- `reasoning` comes before `label`. The model fills in the fields in order, so it writes what the code does and what the docstring says before it chooses the label. W07b's `Review` put `reasoning` first for the same reason.
- The prompt gives triage the code, not only the docstrings. Triage has to tell whether the behavior the reporter describes really happens, and only the code shows that.

In [ ]:
print(team.TRIAGE_PROMPT)
print()
print(json.dumps(team.Label.model_json_schema()["properties"], indent=1))

### TODO 2.1: triage (2 pts)

`bugteam/team.py`: `triage(state)`.

**Steps:**

1. Make the structured model: `config.chat().with_structured_output(Label)`.
2. Build the code part of the prompt: every file in `repo_files()` whose path starts with `"huskyutils/"`, each preceded by a line `# <path>`, joined with blank lines.
3. For each issue in `state["issues"]`, call the model through `retry` with `TRIAGE_PROMPT.format(code=..., issue=issue_text(issue))`. Keep its `label` and its `reasoning`, each in a dictionary keyed by the issue's number.
4. Return `{"labels": labels, "reasons": reasons}`.

In [ ]:
from bugteam.repo import issue_text, repo_files
from bugteam.team import TRIAGE_PROMPT, Label, TeamState
from bugteam.util import retry


# TODO 2.1
def triage(state: TeamState) -> dict:
    # your code here
    ...

In [ ]:
decided = triage({"issues": issues})
for n, label in decided["labels"].items():
    print(f"#{n} {label:12} {decided['reasons'][n]}")

Compare these labels with the ten you wrote down in notebook 00. Where they differ, read the issue and the docstring again and decide who is right. Do not look in `eval/` yet: it holds the answers that notebook 03 scores against, and a triage prompt tuned to them would make that score meaningless.

If triage is wrong on an issue, you may improve `TRIAGE_PROMPT` or the field descriptions, but write down what you changed and why: notebook 03 asks.

## 2. One pair per bug

`fan_out` is the fan-out from W07a.2. It returns one `Send` per issue triage labeled a bug, so that their pairs run in parallel, each with its own sandbox. Each `Send` carries what one pair needs: the issue, and triage's reasoning, which the fixer reads as a hint.

If triage found no bugs at all, there is nothing to fix, and `fan_out` returns `"report"` instead, so that the graph goes straight to the report.

`fix` is given. It runs `fix_issue` from notebook 01 on one payload. If anything goes wrong inside one pair, a sandbox that fails to start or a model call that errors, it records the error for that issue instead of stopping the team, so that one bad issue does not lose the other results.

### TODO 2.2: the fan-out (1 pt)

`bugteam/team.py`: `fan_out(state)`.

**Steps:**

1. Find the issues in `state["issues"]` whose label in `state["labels"]` is `"bug"`.
2. If there are none, return `"report"`.
3. Otherwise return one `Send("fix", {"issue": ..., "note": ...})` per bug, with triage's reasoning for that issue as the note.

In [ ]:
from langgraph.types import Send


# TODO 2.2
def fan_out(state: TeamState):
    # your code here
    ...

In [ ]:
two = issues[:3]
labels = {two[0]["number"]: "bug", two[1]["number"]: "bug", two[2]["number"]: "not_a_bug"}
reasons = {two[0]["number"]: "r1", two[1]["number"]: "r2", two[2]["number"]: "r3"}
sends = fan_out({"issues": two, "labels": labels, "reasons": reasons})
assert [s.node for s in sends] == ["fix", "fix"]
assert sends[0].arg["issue"]["number"] == two[0]["number"] and sends[0].arg["note"] == "r1"
no_bugs = {"issues": two[2:], "labels": labels, "reasons": reasons}
assert fan_out(no_bugs) == "report"
print("the fan-out works")

## 3. The report

The report is what a maintainer reads, so every detail in it has to be true. Read `report` in `team.py`. Most of it is written by code, not by a model: the table, and one section per issue with triage's reasoning, the fix as a diff, and the verifier's evidence and its test. All of these come from what the agents recorded, so the report cannot claim a fix that was not made.

One part is written by a model: a short summary paragraph at the top. `facts(state)` (given) turns each issue's outcome into one line, and the model writes the paragraph from those lines and nothing else.

### TODO 2.3: the summary (1 pt)

`bugteam/team.py`: `summary(state)`.

**Steps:**

1. Join the lines of `facts(state)` with newlines.
2. Fill in `REPORT_PROMPT` with them, call `config.chat().invoke` through `retry`, and return the reply's text, with `text_of`, stripped of surrounding whitespace.

In [ ]:
from bugteam.team import REPORT_PROMPT, facts
from bugteam.util import text_of


# TODO 2.3
def summary(state: TeamState) -> str:
    # your code here
    ...

In [ ]:
example = {"issues": issues[:3], "labels": decided["labels"], "reasons": decided["reasons"],
           "results": [{"number": issues[0]["number"], "status": "verified", "rounds": 2}]}
print("\n".join(facts(example)))
print()
print(summary(example))

Check the paragraph against the lines above it. Does it say anything the lines do not? A summary that adds a detail, even a plausible one, is the kind of error the rest of the report was built to avoid.

## 4. The graph

### TODO 2.4: the team's graph (1 pt)

`bugteam/team.py`: `build_graph()`.

**Steps:**

1. A `StateGraph(TeamState)` with three nodes: `"triage"`, `"fix"`, and `"report"`. Use `team.fix` and `team.report` for the last two; they are given.
2. An edge from `START` to `"triage"`.
3. A conditional edge from `"triage"` with `fan_out`. Give it the list of targets, `["fix", "report"]`, so that the drawing shows both.
4. An edge from `"fix"` to `"report"`, and from `"report"` to `END`. The edge from `"fix"` waits for every pair to finish, and `operator.add` on `results` collects their records, as the workers' results were collected in W07a.2.
5. Compile it and return it.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


# TODO 2.4
def build_graph():
    # your code here
    ...


display(Image(build_graph().get_graph().draw_mermaid_png()))

Your drawing should look like this. The dotted edges from `triage` are the conditional edge: one `Send` to `fix` per bug, or straight to `report` when there are none.

![the team's graph](figures/team_graph.png)

## Before you move on

Move `triage`, `fan_out`, `summary`, and `build_graph` into `bugteam/team.py`. In the module, `build_graph` should use the module's own `triage`, `fan_out`, `fix`, and `report`. `team.run` builds the graph from there, and notebook 03 calls it. Then run the cell below; it makes no model calls.

In [ ]:
importlib.reload(team)
graph = team.build_graph()
assert set(graph.get_graph().nodes) >= {"triage", "fix", "report"}
assert team.fan_out(no_bugs) == "report"
print("the module works")

## 5. A run

### TODO 2.5: run the team and read what it did (1 pt)

The cell below runs the whole team on all ten issues, counts its model calls with `CallCounter`, and saves the run to `results/team.json` and the report to `results/REPORT_team.md`. Notebook 03 reads both, so you do not need to run this again for it.

This is a full run: about 50 to 120 model calls, and one sandbox per verification. Check that your quota allows it before you start. If it stops with a rate-limit error, wait a minute and run it again; it starts over.

Then open `results/REPORT_team.md` and answer the three questions below the cell.

In [ ]:
import time
from bugteam.util import CallCounter

counter, start = CallCounter(), time.time()
out = team.run(issues, callbacks=[counter])
run = {"model": config.model_name(), "sandbox": config.SANDBOX, "max_rounds": config.MAX_ROUNDS,
       "calls": counter.calls, "tokens": counter.tokens, "seconds": round(time.time() - start),
       "labels": out["labels"], "reasons": out["reasons"], "results": out["results"]}

os.makedirs(config.RESULTS, exist_ok=True)
with open(os.path.join(config.RESULTS, "team.json"), "w") as f:
    json.dump(run, f, indent=1)
with open(os.path.join(config.RESULTS, "REPORT_team.md"), "w") as f:
    f.write(out["report"])

print(f"{run['calls']} model calls, {run['tokens']:,} tokens, {run['seconds']} s\n")
for r in sorted(out["results"], key=lambda r: r["number"]):
    print(f"#{r['number']}: {r['status']} after {r['rounds']} verifier run(s)")
    for line in r["log"]:
        print("   -", line[:200].replace("\n", " "))

**Questions** (answer in this cell):

1. Which issues did the verifier send back to the fixer? For one of them, quote the reason, and say whether the next fix addressed it.
2. In the report, pick one fix marked as verified and read the verifier's test. Does it check every example in the function's docstring? If it leaves one out, would the fix have passed that example?
3. Is there anything in the summary paragraph that the table does not support?

*Your answers here.*

## 6. The app

### TODO 2.6: deploy the app (3 pts)

`app.py` serves your team through `bugteam_ui.py`, which is given. You pick an open issue, or write one of your own, and press **Fix**. The app shows triage's label, then the fixer and the verifier handing the fix back and forth as a chat while it happens, and the maintainer's report for that issue when the run finishes.

Run it locally first. In the VS Code terminal, from this folder:

```
python app.py
```

Open http://localhost:7860, fix one issue that is a bug and one that is not, and check that the chat shows the handoffs. Stop it with Ctrl+C.

Then deploy it on Render, as you did in DADA2. `DEPLOY.md` has the steps.

**Put the URL of your deployed app in your README.**